In [1]:
import numpy as np
from scipy.optimize import minimize
import pandas as pd

In [ ]:
raw_data = np.loadtxt("AAPT_test_3sec_20260416.csv", delimiter=",")
raw_coinc = raw_data[:,2]
Input_state_real = np.loadtxt("Initial_state_sample_real.csv", delimiter = ",", dtype = complex)
Input_state_imag = np.loadtxt("Initial_state_sample_imag.csv", delimiter = ",", dtype = complex)
Input_density_matrix = Input_state_real + Input_state_imag*1j

In [3]:
def result_labeling(result):
    """ Label the result data for saving
    result [ndarray]: coincidence data as float numpy array
    return: Dictionary data with polarization stated combination labeled
    """
    labels = ["H", "V", "D", "A", "R", "L"]
    counts_idler_signal = {}
    for ii in labels:
        for ss in labels:
            idler_order = labels.index(ii)
            signal_order = labels.index(ss)
            counts_idler_signal[(ii, ss)] = raw_coinc[idler_order * 6 + signal_order]
    return counts_idler_signal

In [4]:
counts_idler_signal = result_labeling(raw_coinc)
# np.savetxt("Sample_data_mixded_coincidence.csv", counts_idler_signal, delimiter = ',')

In [5]:
def ket_H():
    return np.array([1, 0], dtype=complex)

def ket_V():
    return np.array([0, 1], dtype=complex)

def ket_D():
    return (ket_H() + ket_V()) / np.sqrt(2)

def ket_A():
    return (ket_H() - ket_V()) / np.sqrt(2)

def ket_R():
    return (ket_H() + 1j * ket_V()) / np.sqrt(2)

def ket_L():
    return (ket_H() - 1j * ket_V()) / np.sqrt(2)

KETS_6 = {
    "H": ket_H(),
    "V": ket_V(),
    "D": ket_D(),
    "A": ket_A(),
    "R": ket_R(),
    "L": ket_L(),
}

# def projector(label: str):
def projector(label: str):
    k = KETS_6[label]
    return np.outer(k, np.conjugate(k))

def vec(M: np.ndarray):
    return M.reshape(-1, order="F")

def square_shape(v: np.ndarray, dim: int):
    return v.reshape((dim, dim), order="F")

def pauli_basis():
    I = np.array([[1, 0], [0, 1]], dtype = complex)
    X = np.array([[0, 1], [1, 0]], dtype = complex)
    Y = np.array([[0, -1j], [1j, 0]], dtype = complex)
    Z = np.array([[1, 0], [0, -1]], dtype = complex)
    return [I / np.sqrt(2), X / np.sqrt(2), Y / np.sqrt(2), Z / np.sqrt(2)]

def Meas_ops():
    labels = ["H", "V", "D", "A", "R", "L"]
    meas_labels = []
    meas_ops = []

    for idler_label in labels:
        for signal_label in labels:
            Proj_i = projector(idler_label)
            Proj_s = projector(signal_label)
            M = np.kron(Proj_s, Proj_i)
            meas_labels.append((idler_label, signal_label))
            meas_ops.append(M)

    return meas_labels, meas_ops

In [6]:
# bb = Meas_ops()[0]
# print(np.size(bb[0]))
# print(bb[:])
# print(np.size(cc[0]))
# print(cc[0])

In [7]:
def raw_list2array(raw_data):
    """ Data conversion from state labeled dictionary(dict) to number array
    raw_data (dict): 36 dictionary data for full measurement data
    return: 36 ndarray data
    """
    meas_labels = Meas_ops()[0]
    counts_data_raw = np.array([float(raw_data[label]) for label in meas_labels], dtype = float)
    return counts_data_raw

In [10]:
coinc_data = raw_list2array(counts_idler_signal)
# print(coinc_data)

In [ ]:
def hermitian_initial(rho):
    """ Make positive semidefinite matrix Hermitian 4x4 matrix output, with trace = 1
    rho (): 
    return: 4x4 complex PSD Hermitian matrix
    """
    rho = (rho + rho.conj().T) / 2
    vals, vecs = np.linalg.eigh(rho)
    vals = np.clip(vals, 0.0, None)
    if np.sum(vals) <= 0:
        return np.eye(rho.shape[0], dtype = complex) / rho.shape[0]
    rho_psd = vecs @ np.diag(vals) @ vecs.conj().T
    rho_psd_real = np.real(rho_psd)
    rho_psd_imag = np.imag(rho_psd)*1e9
    rho_psd_imag_int = rho_psd_imag.astype(int)
    rho_psd_imag_diag = rho_psd_imag_int.astype(complex)
    rho_psd_imag_diag /= 1e9
    rho_psd_recon = rho_psd_real + rho_psd_imag_diag * 1j
    rho_psd_recon /= np.trace(rho_psd_recon)
    return rho_psd_recon

def linear_inversion_initial(counts_raw):
    """ Create non-optimized 4x4 output state density matrix from raw counts data
    counts_raw (): 
    """
    meas_labels, meas_ops = Meas_ops()

    # counts = np.array([float(counts_raw[label]) for label in meas_labels], dtype = float)
    total = np.sum(counts_raw) / 9 # for 36 coincidence counts data, devided by 9 (each set has four combinations)

    if total <= 0:
        raise ValueError("Total counts must be positive.")

    probs = counts_raw / total

    A = np.vstack([vec(M).conj() for M in meas_ops])
    rho_vec = np.linalg.lstsq(A, probs, rcond = None)[0]
    rho = square_shape(rho_vec, 4)

    # rho = (rho + rho.conj().T) / 2
    # tr = np.trace(rho)
    # if abs(tr) < 1e-15:
    #     rho = np.eye(4, dtype = complex) / 4
    # else:
    #     rho = rho / tr

    rho = hermitian_initial(rho)
    return rho

In [56]:
meas_labels, meas_ops = Meas_ops()
A = np.vstack([vec(M).conj() for M in meas_ops]) 
print(A[0])
print(A.shape)

[1.-0.j 0.-0.j 0.-0.j 0.-0.j 0.-0.j 0.-0.j 0.-0.j 0.-0.j 0.-0.j 0.-0.j
 0.-0.j 0.-0.j 0.-0.j 0.-0.j 0.-0.j 0.-0.j]
(36, 16)


In [50]:
qqq = linear_inversion_initial(coinc_data)
print(qqq)

[[ 0.09982027+0.j          0.10882241-0.14846882j  0.04402286+0.18892826j
  -0.08095535-0.02646772j]
 [ 0.10882241+0.14846882j  0.40147652+0.j         -0.26072535+0.29117439j
  -0.04313195-0.17964084j]
 [ 0.04402286-0.18892826j -0.26072535-0.29117439j  0.4044327 +0.j
  -0.10452226+0.14980538j]
 [-0.08095535+0.02646772j -0.04313195+0.17964084j -0.10452226-0.14980538j
   0.09427051+0.j        ]]


In [48]:
qqq = linear_inversion_initial(coinc_data)
print(qqq)
# qqqq = np.trace(qqq)
# print(np.real(qqq))
# print(np.imag(qqq))
# www = np.imag(qqq)*1e9
# print(www)
# wwww = www.astype(int)
# print(wwww)
# wwwww = wwww.astype(float)
# wwwww /= 1e9
# print(wwwww)

# eeeee = np.real(qqq) + 1j*wwwww
# print(eeeee)

[[ 0.09982027+0.j          0.10882241-0.14846882j  0.04402286+0.18892826j
  -0.08095535-0.02646772j]
 [ 0.10882241+0.14846882j  0.40147652+0.j         -0.26072535+0.29117439j
  -0.04313195-0.17964084j]
 [ 0.04402286-0.18892826j -0.26072535-0.29117439j  0.4044327 +0.j
  -0.10452226+0.14980538j]
 [-0.08095535+0.02646772j -0.04313195+0.17964084j -0.10452226-0.14980538j
   0.09427051+0.j        ]]


In [44]:
def lower_tri_complex(params, dim = 4):
    T = np.zeros((dim, dim), dtype=complex)

    for i in range(dim):
        T[i, i] = params[i]

    idx = dim
    for i in range(1, dim):
        for j in range(i):
            T[i, j] = params[idx] + 1j * params[idx + 1]
            idx += 2

    return T

def chi_recon_unscaled(params):
    T = lower_tri_complex(params, dim = 4)
    chi = T.conj().T @ T
    chi = (chi + chi.conj().T) / 2
    return chi

In [ ]:
# ============================================================
# 5. Apply channel on Signal only
# ============================================================
def output_state_recon(rho_in, chi, basis = 'Pauli'):
    if basis == 'Pauli':
        chan_basis = pauli_basis()
    else:
        chan_basis = basis

    I_idler = np.eye(2, dtype=complex)
    rho_out = np.zeros((4, 4), dtype=complex)

    for m, Fm in enumerate(chan_basis):
        for n, Fn in enumerate(chan_basis):
            Km = np.kron(Fm, I_idler)
            Kn = np.kron(Fn, I_idler)
            rho_out += chi[m, n] * Km @ rho_in @ Kn.conj().T

    rho_out = (rho_out + rho_out.conj().T) / 2
    return rho_out

In [ ]:
# ============================================================
# 6. TNI constraint
#    sum chi_mn F_n^\dagger F_m <= I
# ============================================================

def tni_matrix(chi, basis = 'Pauli'):
    if basis == 'Pauli':
        chan_basis = pauli_basis()
    else:
        chan_basis = basis

    K = np.zeros((2, 2), dtype = complex)
    for m, Fm in enumerate(chan_basis):
        for n, Fn in enumerate(chan_basis):
            K += chi[m, n] * Fn.conj().T @ Fm

    return np.eye(2, dtype = complex) - K

def cp_penalty(H):
    H = (H + H.conj().T) / 2
    evals = np.linalg.eigvalsh(H)
    neg_parts = np.clip(-evals, 0.0, None)
    return np.sum(neg_parts**2)

In [ ]:
# ============================================================
# 7. Poisson likelihood
# ============================================================

def poisson_log_likeli(raw_counts, expected_counts, eps = 1e-12):
    mu = np.clip(expected_counts, eps, None)
    n = raw_counts
    return np.sum(mu - n * np.log(mu))

In [ ]:
# ============================================================
# 8. Main estimator using 36 real coincidence counts
# ============================================================

def eapt_ntp(
    rho_in,
    counts_idler_signal,
    integration_times = None,
    backgrounds = None,
    source_scale_mode = "fit_global_scale",
    fixed_global_scale = None,
    tni_penalty = 1e6,
    positivity_eps = 1e-12,
    basis = 'Pauli'
):
    # """
    # Estimate NTP chi matrix from:
    #   - known input bipartite density matrix rho_in
    #   - 36 measured coincidence counts
    #   - Poisson likelihood
    #   - TNI penalty

    # Parameters
    # ----------
    # rho_in : np.ndarray
    #     4x4 complex density matrix, ordering Signal ⊗ Idler
    # counts_idler_signal : dict
    #     Keys: (idler_label, signal_label), labels in {"H","V","D","A","R","L"}
    #     Values: observed coincidence counts
    # integration_times : dict or None
    #     Same keys as counts, values are acquisition times
    # backgrounds : dict or None
    #     Same keys as counts, values are known background counts
    # source_scale_mode : str
    #     "fit_global_scale" or "fixed_global_scale"
    # fixed_global_scale : float or None
    #     Required if source_scale_mode == "fixed_global_scale"
    # tni_penalty : float
    #     Penalty weight for TNI constraint
    # positivity_eps : float
    #     Numerical floor for expected counts

    # Returns
    # -------
    # chi_est : np.ndarray
    # rho_out_tilde_est : np.ndarray
    # result : scipy OptimizeResult
    # diagnostics : dict
    # """
    rho_in = np.array(rho_in, dtype = complex)
    if rho_in.shape != (4, 4):
        raise ValueError("rho_in must be a 4x4 complex matrix.")

    rho_in = (rho_in + rho_in.conj().T) / 2
    tr_in = np.trace(rho_in)
    if abs(tr_in) < 1e-15:
        raise ValueError("rho_in has near-zero trace.")
    rho_in = rho_in / tr_in

    meas_labels, meas_ops = Meas_ops()
    if basis == 'Pauli':
        chan_basis = pauli_basis()
    else:
        chan_basis = basis

    # Convert input dicts to arrays in fixed internal order
    observed_counts = np.array(
        [float(counts_idler_signal[label]) for label in meas_labels],
        dtype=float
    )
    if np.any(observed_counts < 0):
        raise ValueError("Observed counts must be nonnegative.")

    if integration_times is None:
        integration_times = {label: 1.0 for label in meas_labels}
    if backgrounds is None:
        backgrounds = {label: 0.0 for label in meas_labels}

    times = np.array([float(integration_times[label]) for label in meas_labels], dtype=float)
    bkg = np.array([float(backgrounds[label]) for label in meas_labels], dtype=float)

    if np.any(times <= 0):
        raise ValueError("All integration times must be positive.")
    if np.any(bkg < 0):
        raise ValueError("Background counts must be nonnegative.")

    fit_global_scale = (source_scale_mode == "fit_global_scale")
    if source_scale_mode not in ("fit_global_scale", "fixed_global_scale"):
        raise ValueError("source_scale_mode must be 'fit_global_scale' or 'fixed_global_scale'.")
    if (not fit_global_scale) and (fixed_global_scale is None):
        raise ValueError("fixed_global_scale must be provided when using fixed_global_scale mode.")

    def unpack_params(x):
        chi_params = x[:16]
        if fit_global_scale:
            log_s = x[16]
            global_scale = np.exp(log_s)
        else:
            global_scale = float(fixed_global_scale)
        return chi_params, global_scale

    def objective(x):
        chi_params, global_scale = unpack_params(x)
        chi = chi_recon_unscaled(chi_params)

        rho_out_tilde = output_state_recon(rho_in, chi, chan_basis)

        signal_expectations = np.array(
            [np.real(np.trace(M @ rho_out_tilde)) for M in meas_ops],
            dtype=float
        )
        signal_expectations = np.clip(signal_expectations, 0.0, None)

        mu = global_scale * times * signal_expectations + bkg

        nll = poisson_log_likeli(observed_counts, mu, eps=positivity_eps)

        R = tni_matrix(chi, chan_basis)
        tni_cost = tni_penalty * cp_penalty(R)

        return nll + tni_cost

    # Initial guess
    x0 = np.zeros(17 if fit_global_scale else 16, dtype=float)

    # Identity-like but small
    x0[0] = 0.7
    x0[1] = 0.1
    x0[2] = 0.1
    x0[3] = 0.1

    if fit_global_scale:
        rough_scale = max(np.sum(observed_counts - bkg), 1.0)
        x0[16] = np.log(rough_scale)

    result = minimize(
        objective,
        x0,
        method="L-BFGS-B"
    )

    chi_params_opt, global_scale_opt = unpack_params(result.x)
    chi_est = chi_recon_unscaled(chi_params_opt)
    rho_out_tilde_est = output_state_recon(rho_in, chi_est, chan_basis)

    expected_signal = np.array(
        [np.real(np.trace(M @ rho_out_tilde_est)) for M in meas_ops],
        dtype=float
    )
    expected_signal = np.clip(expected_signal, 0.0, None)
    expected_counts = global_scale_opt * times * expected_signal + bkg

    diagnostics = {
        "global_scale": global_scale_opt,
        "measurement_labels": meas_labels,
        "observed_counts": observed_counts,
        "expected_counts": expected_counts,
        "expected_signal_subnormalized": expected_signal,
        "subnormalized_trace_rho_out": np.real(np.trace(rho_out_tilde_est)),
        "tni_residual_matrix": tni_matrix(chi_est, chan_basis),
        "tni_penalty_value": cp_penalty(tni_matrix(chi_est, chan_basis)),
        "objective_value": result.fun,
        "success": result.success,
        "message": result.message,
    }

    return chi_est, rho_out_tilde_est, result, diagnostics

In [ ]:
# ============================================================
# 9. Optional helpers
# ============================================================

def chi_to_unnormalized_pauli(chi_orth):
    # """
    # Convert from orthonormal basis {I,X,Y,Z}/sqrt(2)
    # to unnormalized basis {I,X,Y,Z}.
    # """
    return chi_orth / 2.0

def channel_effect_operator(chi, basis=None):
    # """
    # K = sum chi_mn F_n^\dagger F_m
    # TP:  K = I
    # TNI: K <= I
    # """
    if basis is None:
        basis = pauli_basis()

    K = np.zeros((2, 2), dtype=complex)
    for m, Fm in enumerate(basis):
        for n, Fn in enumerate(basis):
            K += chi[m, n] * Fn.conj().T @ Fm
    return K

def print_tni_diagnostics(chi):
    K = channel_effect_operator(chi)
    R = np.eye(2, dtype=complex) - K
    evals_R = np.linalg.eigvalsh((R + R.conj().T) / 2)

    print("K = sum chi_mn F_n^dag F_m :")
    print(K)
    print("\nR = I - K :")
    print(R)
    print("\nEigenvalues of R (should be >= 0 for TNI):")
    print(evals_R)


In [ ]:
# ============================================================
# 10. Example input format
# ============================================================

if __name__ == "__main__":
    np.set_printoptions(precision=6, suppress=True)

    # Example rho_in: ideal Psi+ in basis [HH, HV, VH, VV]
    psi_plus = np.array([0, 1, 1, 0], dtype=complex) / np.sqrt(2)
    rho_in = np.outer(psi_plus, psi_plus.conj())

    # Replace these with your real experimental 36 counts
    labels = ["H", "V", "D", "A", "R", "L"]
    counts = {}
    for i in labels:
        for s in labels:
            counts[(i, s)] = 1000.0

    # Optional per-setting times / backgrounds
    times = {key: 1.0 for key in counts.keys()}
    backgrounds = {key: 0.0 for key in counts.keys()}

    chi_est, rho_out_tilde_est, result, diagnostics = eapt_ntp(
        rho_in=rho_in,
        counts_idler_signal=counts,
        integration_times=times,
        backgrounds=backgrounds,
        source_scale_mode="fit_global_scale",
        tni_penalty=1e6
    )

    print("Optimization success:", result.success)
    print("Message:", result.message)
    print("Final objective value:", diagnostics["objective_value"])
    print("Estimated global scale:", diagnostics["global_scale"])

    print("\nEstimated chi matrix in orthonormal Pauli basis {I,X,Y,Z}/sqrt(2):")
    print(chi_est)

    print("\nEstimated chi matrix in unnormalized Pauli basis {I,X,Y,Z}:")
    print(chi_to_unnormalized_pauli(chi_est))

    print("\nEstimated subnormalized output state rho_out_tilde:")
    print(rho_out_tilde_est)

    print("\nTrace of rho_out_tilde (survival probability for this rho_in):")
    print(diagnostics["subnormalized_trace_rho_out"])

    print("\nObserved vs expected counts:")
    for label, n_obs, n_exp in zip(
        diagnostics["measurement_labels"],
        diagnostics["observed_counts"],
        diagnostics["expected_counts"]
    ):
        print(f"{label}: observed={n_obs:.3f}, expected={n_exp:.3f}")

    print()
    print_tni_diagnostics(chi_est)